# MILP v10 — Bridge Comparison: v1 (spec v3) vs v7

Runs the MILP v10 mainline case (M2_I1_R0) with both bridge packages,
then compares the resulting optimal sizing and costs.

In [1]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from milp_common import get_config, load_data, CASE_TABLE, format_results

# We import the model builder from the cases notebook via exec
# (it's defined inline there, so we replicate it here)
import gurobipy as gp
from gurobipy import GRB
import time

print('Imports OK')

Imports OK


In [2]:
def build_and_solve(CFG, repday_data, calendar_order, info, case_flags):
    """Build and solve the v10 MILP for one case (same as milp_v10_cases)."""
    use_method1 = case_flags.get('method1', True) and calendar_order is not None
    n_repdays = info['n_repdays']
    n_scenarios = info['n_scenarios']
    n_hours = info['n_hours']
    N = n_repdays * n_scenarios * n_hours
    PV_RATING = CFG['pv_rating_kw']
    
    pv_coeff = np.empty(N); load_arr = np.empty(N)
    pw_arr = np.empty(N); tou_arr = np.empty(N); month_arr = np.empty(N, dtype=int)
    
    def flat(d, s, t): return d * (n_scenarios * n_hours) + s * n_hours + t
    
    for d in range(n_repdays):
        dd = repday_data[d]
        for s in range(n_scenarios):
            sc = dd['scenarios'][s]
            start = flat(d, s, 0)
            sl = slice(start, start + n_hours)
            pv_coeff[sl] = sc['pv_kw'] / PV_RATING
            load_arr[sl] = sc['load_kw']
            pw_arr[sl] = sc['prob'] * dd['weight']
            tou_arr[sl] = dd['tou']
            month_arr[sl] = dd['month']
    
    is_first = np.zeros(N, dtype=bool)
    for d in range(n_repdays):
        for s in range(n_scenarios):
            is_first[flat(d, s, 0)] = True
    first_hours = np.where(is_first)[0]
    later_hours = np.where(~is_first)[0]
    prev_idx_arr = np.arange(N) - 1
    
    m = gp.Model('milp_v10')
    m.Params.OutputFlag = 0
    m.Params.TimeLimit = CFG['time_limit']
    m.Params.Method = 2; m.Params.Presolve = 2; m.Params.Cuts = 2
    m.Params.MIPGap = 1e-4; m.Params.MIPFocus = 1
    
    cap_pv = m.addVar(ub=CFG['pv_max_kw'], name='cap_pv')
    cap_bp = m.addVar(ub=CFG['bess_p_max_kw'], name='cap_bp')
    cap_be = m.addVar(ub=CFG['bess_e_max_kwh'], name='cap_be')
    cap_cd = m.addVar(name='cap_cd')
    
    p_grid = m.addMVar(N, name='grid'); p_pv_load = m.addMVar(N, name='pv_load')
    p_pv_ch = m.addMVar(N, name='pv_ch'); p_curt = m.addMVar(N, name='curt')
    p_ch = m.addMVar(N, name='ch'); p_dis = m.addMVar(N, name='dis')
    bin_ch = m.addMVar(N, vtype=GRB.BINARY, name='bch')
    bin_dis = m.addMVar(N, vtype=GRB.BINARY, name='bdis')
    green_e = m.addMVar(N, name='green_e'); green_dis = m.addMVar(N, name='green_dis')
    delta_e_cum = m.addMVar(N, lb=-GRB.INFINITY, name='dec')
    trec_buy = m.addVar(name='trec')
    m.update()
    
    eff_ch = CFG['eff_charge']; eff_dis_inv = 1.0 / CFG['eff_discharge']
    
    m.addConstrs((p_pv_load[i] + p_grid[i] + p_dis[i] == load_arr[i] + p_ch[i] for i in range(N)), name='bal')
    m.addConstrs((p_pv_load[i] + p_pv_ch[i] + p_curt[i] == pv_coeff[i] * cap_pv for i in range(N)), name='pv')
    m.addConstrs((p_pv_ch[i] <= p_ch[i] for i in range(N)), name='pvch')
    m.addConstrs((p_ch[i] <= cap_bp * bin_ch[i] for i in range(N)), name='chl')
    m.addConstrs((p_dis[i] <= cap_bp * bin_dis[i] for i in range(N)), name='disl')
    m.addConstrs((bin_ch[i] + bin_dis[i] <= 1 for i in range(N)), name='excl')
    
    m.addConstrs((delta_e_cum[i] == eff_ch * p_ch[i] - eff_dis_inv * p_dis[i] for i in first_hours), name='di')
    m.addConstrs((delta_e_cum[i] == delta_e_cum[prev_idx_arr[i]] + eff_ch * p_ch[i] - eff_dis_inv * p_dis[i] for i in later_hours), name='dc')
    
    if use_method1:
        n_cal = len(calendar_order)
        E_inter = m.addMVar(n_cal, lb=0, name='Eint')
        for ci, cd in enumerate(calendar_order):
            d = cd['d_idx']
            for s in range(n_scenarios):
                for t in range(n_hours):
                    fi = flat(d, s, t)
                    m.addConstr(E_inter[ci] + delta_e_cum[fi] >= CFG['soc_min'] * cap_be)
                    m.addConstr(E_inter[ci] + delta_e_cum[fi] <= CFG['soc_max'] * cap_be)
        for ci in range(n_cal - 1):
            d = calendar_order[ci]['d_idx']
            ede = gp.LinExpr()
            for s in range(n_scenarios):
                ede += repday_data[d]['scenarios'][s]['prob'] * delta_e_cum[flat(d, s, n_hours-1)]
            m.addConstr(E_inter[ci+1] == E_inter[ci] + ede)
        eps = CFG['epsilon_term']
        dl = calendar_order[-1]['d_idx']
        edl = gp.LinExpr()
        for s in range(n_scenarios):
            edl += repday_data[dl]['scenarios'][s]['prob'] * delta_e_cum[flat(dl, s, n_hours-1)]
        m.addConstr(E_inter[n_cal-1] + edl - E_inter[0] <= eps * cap_be)
        m.addConstr(E_inter[n_cal-1] + edl - E_inter[0] >= -eps * cap_be)
        m.addConstr(E_inter[0] == CFG['soc_init'] * cap_be)
    else:
        for i in range(N):
            m.addConstr(CFG['soc_init'] * cap_be + delta_e_cum[i] >= CFG['soc_min'] * cap_be)
            m.addConstr(CFG['soc_init'] * cap_be + delta_e_cum[i] <= CFG['soc_max'] * cap_be)
    
    gf = 0.5 * CFG['soc_init']
    m.addConstrs((green_e[i] == gf * cap_be + eff_ch * p_pv_ch[i] - green_dis[i] for i in first_hours), name='gi')
    m.addConstrs((green_e[i] == green_e[prev_idx_arr[i]] + eff_ch * p_pv_ch[i] - green_dis[i] for i in later_hours), name='gc')
    m.addConstrs((green_dis[i] <= p_dis[i] for i in range(N)), name='gdl')
    m.addConstrs((green_e[i] <= CFG['soc_max'] * cap_be for i in range(N)), name='gec')
    
    months_in = sorted(set(month_arr))
    D_max = {mo: m.addVar(name=f'Dm{mo}') for mo in months_in}
    oc1 = {mo: m.addVar(name=f'o1{mo}') for mo in months_in}
    oc2 = {mo: m.addVar(name=f'o2{mo}') for mo in months_in}
    kappa = CFG['kappa']
    for i in range(N): m.addConstr(D_max[int(month_arr[i])] >= kappa * p_grid[i])
    for mo in months_in:
        m.addConstr(oc1[mo] >= D_max[mo] - cap_cd)
        m.addConstr(oc1[mo] <= 0.10 * cap_cd)
        m.addConstr(oc2[mo] >= D_max[mo] - 1.10 * cap_cd)
    
    total_load = float(np.sum(pw_arr * load_arr))
    re_expr = gp.quicksum(pw_arr[i] * (p_pv_load[i] + green_dis[i]) for i in range(N))
    m.addConstr(re_expr + trec_buy >= CFG['re_target'] * total_load, name='RE20')
    
    inv = (cap_pv * CFG['capex_pv_per_kw'] * CFG['crf_pv']
        + cap_bp * CFG['capex_bess_power_per_kw'] * CFG['crf_bess']
        + cap_be * (CFG['capex_bess_energy_per_kwh'] * CFG['crf_bess'] + CFG['capex_bess_energy_per_kwh'] * CFG['fom_bess_rate'])
        + cap_cd * CFG['contract_price_per_kw_month'] * 12)
    gc = gp.quicksum(pw_arr[i] * p_grid[i] * float(tou_arr[i]) for i in range(N))
    bp = gp.quicksum(oc1[mo] * CFG['contract_price_per_kw_month'] * CFG['oc_within_10pct_mult'] + oc2[mo] * CFG['contract_price_per_kw_month'] * CFG['oc_beyond_10pct_mult'] for mo in months_in)
    dg = gp.quicksum(pw_arr[i] * 0.05 * (p_ch[i] + p_dis[i]) for i in range(N))
    tc = CFG['trec_cost_per_kwh'] * trec_buy
    m.setObjective(inv + gc + bp + dg + tc, GRB.MINIMIZE)
    
    t0 = time.time()
    m.optimize()
    st = time.time() - t0
    
    if m.Status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL): return None
    
    re_val = sum(pw_arr[i] * (p_pv_load[i].X + green_dis[i].X) for i in range(N))
    return {
        'cap_pv': cap_pv.X, 'cap_bp': cap_bp.X, 'cap_be': cap_be.X, 'cap_cd': cap_cd.X,
        'obj': m.ObjVal, 're_pct': re_val / total_load * 100,
        'trec_cost': trec_buy.X * CFG['trec_cost_per_kwh'], 'solve_time': st, 'gap': m.MIPGap,
    }

print('Model builder loaded')

Model builder loaded


## Run Mainline (M2_I1_R0) with Both Bridge Packages

In [3]:
mainline = {'method1': True, 'risk_days': True, 'prob_pv': True, 'uplift': None}

# ── Bridge v1 (old: 95 repdays, 20 body + 75 risk) ──
print('='*60)
print('  Bridge v1 (spec v3): 20 body + 75 risk = 95 repdays')
print('='*60)
CFG_v1 = get_config()
CFG_v1['bridge_dir'] = '../bridge_outputs_v1'
rd_v1, co_v1, info_v1 = load_data(CFG_v1, mainline)
res_v1 = build_and_solve(CFG_v1, rd_v1, co_v1, info_v1, mainline)
print(f'  Result: PV={res_v1["cap_pv"]:,.0f}, BESS={res_v1["cap_bp"]:,.0f}/{res_v1["cap_be"]:,.0f}, '
      f'Contract={res_v1["cap_cd"]:,.0f}, Cost={res_v1["obj"]/1e6:.2f}M, RE={res_v1["re_pct"]:.1f}%')

# ── Bridge v7 (new: 44 repdays, 16 body + 28 risk) ──
print('\n' + '='*60)
print('  Bridge v7: 16 body + 28 risk = 44 repdays')
print('='*60)
CFG_v7 = get_config()
CFG_v7['bridge_dir'] = '../bridge_outputs'
rd_v7, co_v7, info_v7 = load_data(CFG_v7, mainline)
res_v7 = build_and_solve(CFG_v7, rd_v7, co_v7, info_v7, mainline)
print(f'  Result: PV={res_v7["cap_pv"]:,.0f}, BESS={res_v7["cap_bp"]:,.0f}/{res_v7["cap_be"]:,.0f}, '
      f'Contract={res_v7["cap_cd"]:,.0f}, Cost={res_v7["obj"]/1e6:.2f}M, RE={res_v7["re_pct"]:.1f}%')

  Bridge v1 (spec v3): 20 body + 75 risk = 95 repdays
CRF PV (20yr, r=0.05): 0.0802
CRF BESS (15yr, r=0.05): 0.0963
  Repdays: 95 (20 body + 75 risk)
  Scenarios: 5/repday, 24 hours
  Weight sum: 365
  Calendar days: 365
Set parameter WLSAccessID


Set parameter WLSSecret


Set parameter LicenseID to value 2797924


Academic license 2797924 - for non-commercial use only - registered to m1___@mail.ntust.edu.tw


  Result: PV=7,168, BESS=1,424/8,071, Contract=2,991, Cost=83.23M, RE=39.0%

  Bridge v7: 16 body + 28 risk = 44 repdays
CRF PV (20yr, r=0.05): 0.0802
CRF BESS (15yr, r=0.05): 0.0963
  Repdays: 44 (16 body + 28 risk)
  Scenarios: 5/repday, 24 hours
  Weight sum: 365
  Calendar days: 365


  Result: PV=6,711, BESS=1,350/7,899, Contract=3,027, Cost=83.52M, RE=36.2%


In [4]:
# ── Comparison Table ──
comp = pd.DataFrame([
    format_results('Bridge_v1 (95 repdays)', res_v1['cap_pv'], res_v1['cap_bp'], res_v1['cap_be'],
                   res_v1['cap_cd'], res_v1['obj'], res_v1['re_pct'], res_v1['trec_cost'],
                   res_v1['solve_time'], CFG_v1),
    format_results('Bridge_v7 (44 repdays)', res_v7['cap_pv'], res_v7['cap_bp'], res_v7['cap_be'],
                   res_v7['cap_cd'], res_v7['obj'], res_v7['re_pct'], res_v7['trec_cost'],
                   res_v7['solve_time'], CFG_v7),
])
print(comp.to_string(index=False))
comp.to_csv('../milp_outputs/bridge_comparison.csv', index=False)
print('\nSaved to milp_outputs/bridge_comparison.csv')

                  case  pv_kw  bess_p_kw  bess_e_kwh  ep_ratio  contract_kw  total_cost_M  invest_M  opex_M  trec_M  re_pct  solve_s
Bridge_v1 (95 repdays) 7168.1     1423.8      8070.9       5.7       2990.7         83.23     39.31   43.92     0.0    39.0     33.7
Bridge_v7 (44 repdays) 6711.1     1350.0      7899.0       5.9       3026.9         83.52     37.72   45.80     0.0    36.2     25.5

Saved to milp_outputs/bridge_comparison.csv
